# AIC 2026 — faster-whisper medium pilot worker
Notebook chạy một trong hai shard ASR pilot. Colab dùng shard 0 (48 video), Kaggle dùng shard 1 (47 video). Luôn smoke 5 video vào output riêng trước full shard.

In [ ]:
from pathlib import Path

WORKER = "colab"  # đổi thành "kaggle" trên Kaggle
NUM_SHARDS = 2
SHARD_INDEX = 0 if WORKER == "colab" else 1
SMOKE_LIMIT = 5
RUN_FULL_SHARD = False  # chỉ bật sau khi smoke validate và transcript đọc được

REPO_DIR = Path("/content/aic2026") if WORKER == "colab" else Path("/kaggle/working/aic2026")
BTC_ROOT = Path("/content/drive/MyDrive/AIC_2026") if WORKER == "colab" else Path("/kaggle/input/aic2026/AIC_2026")
SHARD_DIR = Path("/content/shards") if WORKER == "colab" else Path("/kaggle/input/aic-asr-shards")
RUNTIME_ROOT = Path("/content/asr-worker") if WORKER == "colab" else Path("/kaggle/working/asr-worker")
SHARD_NAME = f"shard_{SHARD_INDEX:03d}_of_{NUM_SHARDS:03d}"
SHARD_FILE = SHARD_DIR / f"{SHARD_NAME}.json"
print({"worker": WORKER, "shard": str(SHARD_FILE), "btc_root": str(BTC_ROOT)})

## Mount/copy repo, BTC root và shard
Copy/upload repo và đúng shard JSON tới các path đã chọn. Colab có thể mount Drive; Kaggle attach inputs dưới `/kaggle/input`.

In [ ]:
if WORKER == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
assert (REPO_DIR / "pyproject.toml").is_file(), f"Thiếu repo: {REPO_DIR}"
assert SHARD_FILE.is_file(), f"Thiếu shard: {SHARD_FILE}"
assert BTC_ROOT.is_dir(), f"Sai BTC_ROOT: {BTC_ROOT}"

## Install repository và ASR extra

In [ ]:
import subprocess
import sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", f"{REPO_DIR}[asr]"])

## Kiểm tra GPU và faster-whisper

In [ ]:
import importlib.metadata
import torch
print("torch CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("faster-whisper:", importlib.metadata.version("faster-whisper"))
assert torch.cuda.is_available(), "Không cố chạy pilot full shard trên CPU"

## Tạo runtime config (không commit)

In [ ]:
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = RUNTIME_ROOT / "aic-runtime.toml"
config_text = (
    '[environment]\nname = "asr-worker"\n\n'
    + '[paths]\n'
    + f'data_root = "{BTC_ROOT.as_posix()}"\n'
    + f'work_root = "{(RUNTIME_ROOT / "work").as_posix()}"\n'
    + f'artifact_root = "{(RUNTIME_ROOT / "artifacts").as_posix()}"\n'
)
CONFIG_PATH.write_text(config_text, encoding="utf-8")
print(CONFIG_PATH.read_text(encoding="utf-8"))

## Smoke 5 video — output riêng

In [ ]:
SMOKE_OUT = RUNTIME_ROOT / "artifacts" / f"smoke_{SHARD_NAME}_limit{SMOKE_LIMIT}"
smoke_cmd = [
    sys.executable, "-m", "aic2026.cli", "run-asr-shard",
    "--config", str(CONFIG_PATH), "--shard", str(SHARD_FILE),
    "--out-dir", str(SMOKE_OUT), "--model", "medium",
    "--model-revision", "medium", "--language", "vi",
    "--beam-size", "5", "--vad-filter", "--device", "auto",
    "--limit-videos", str(SMOKE_LIMIT),
]
subprocess.check_call(smoke_cmd, cwd=REPO_DIR)
subprocess.check_call([sys.executable, "-m", "aic2026.cli", "validate-asr-shard", "--artifact-dir", str(SMOKE_OUT)], cwd=REPO_DIR)

## Xem nhanh 5 transcript segment

In [ ]:
import json
segment_lines = (SMOKE_OUT / "asr_segments.jsonl").read_text(encoding="utf-8").splitlines()
segments = [json.loads(line) for line in segment_lines if line.strip()]
for row in segments[:5]:
    print(row["video_id"], row["start_sec"], row["end_sec"], row["text"])
print("segments:", len(segments))

## Full shard — chỉ bật sau GO smoke

In [ ]:
FULL_OUT = RUNTIME_ROOT / "artifacts" / SHARD_NAME
if RUN_FULL_SHARD:
    full_cmd = [
        sys.executable, "-m", "aic2026.cli", "run-asr-shard",
        "--config", str(CONFIG_PATH), "--shard", str(SHARD_FILE),
        "--out-dir", str(FULL_OUT), "--model", "medium",
        "--model-revision", "medium", "--language", "vi",
        "--beam-size", "5", "--vad-filter", "--device", "auto",
    ]
    subprocess.check_call(full_cmd, cwd=REPO_DIR)
    subprocess.check_call([sys.executable, "-m", "aic2026.cli", "validate-asr-shard", "--artifact-dir", str(FULL_OUT)], cwd=REPO_DIR)
else:
    print("RUN_FULL_SHARD=False: chưa chạy full shard")

## Zip artifact đã validate để tải về local

In [ ]:
import shutil
EXPORT_DIR = RUNTIME_ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_OUT = FULL_OUT if RUN_FULL_SHARD else SMOKE_OUT
assert (SOURCE_OUT / "DONE.json").is_file(), "Chỉ zip artifact đã có DONE.json"
zip_path = shutil.make_archive(str(EXPORT_DIR / SOURCE_OUT.name), "zip", root_dir=SOURCE_OUT.parent, base_dir=SOURCE_OUT.name)
print("Download:", zip_path)